# Homework: Inverted Pendulum World Model Training

This notebook runs the official homework workflow: install dependencies, generate MuJoCo ground-truth windows, train the starter world model, evaluate locked nMSE/VPT scoreboard metrics, and plot diagnostics.

The assignment trains **one dynamics world model**. It does not train a policy, does not use reward, and does not use actor-critic updates.

The public scoreboard is for debugging and comparison. Official grading uses TA-private data generated with hidden seeds.

## 1. Configure Repository

If you are running this notebook inside a local clone, the setup cell will reuse the current repository. Otherwise, set `COURSE_REPO_URL` to your fork and the notebook will clone it.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

COURSE_REPO_URL = "https://github.com/WeijieLai1024/EEC289A_WorldModel-Homework.git"
COURSE_REPO_BRANCH = "main"
SMOKE_RUN = False  # Full public profile by default.
RUN_SCOREBOARD_DEMO = True  # Generate/evaluate configs/public_scoreboard.yaml windows by default.

def run(cmd, cwd=None):
    cmd = [str(part) for part in cmd]
    print("+", " ".join(cmd))
    return subprocess.run(cmd, cwd=cwd, check=True)

def find_existing_repo(start: Path):
    for path in [start, *start.parents]:
        if (path / "wm_hw").is_dir() and (path / "configs" / "dev.yaml").exists():
            return path
    return None

def default_clone_dir():
    content = Path("/content")
    if content.exists() and os.access(content, os.W_OK):
        return content / "wm_inverted_pendulum_hw"
    return Path.home() / "wm_inverted_pendulum_hw"

repo = find_existing_repo(Path.cwd())
if repo is None:
    repo = default_clone_dir()
    if not (repo / ".git").exists():
        run(["git", "clone", "--branch", COURSE_REPO_BRANCH, COURSE_REPO_URL, repo])
    else:
        print(f"Using existing repo at {repo}")

COURSE_REPO_DIR = repo.resolve()
print("COURSE_REPO_DIR =", COURSE_REPO_DIR)

## 2. Install Dependencies And Verify MuJoCo

In [ ]:
%cd {COURSE_REPO_DIR}
!{sys.executable} -m pip install -q -U pip wheel "setuptools<82"
!{sys.executable} -m pip install -q -r requirements.txt

import gymnasium as gym
import torch

env = gym.make("InvertedPendulum-v5", reset_noise_scale=0.01)
obs, info = env.reset(seed=0)
print("obs shape:", obs.shape)
print("action shape:", env.action_space.shape)
print("action range:", env.action_space.low, env.action_space.high)
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
env.close()

## 3. Run Tests

These tests check the environment, dataset schema, no-leak open-loop rollout, differentiable loss, locked VPT/nMSE metrics, and train/eval smoke path.

In [ ]:
!{sys.executable} -m pytest -q -m "not slow"

## 4. Generate MuJoCo Ground-Truth Windows

In [ ]:
DATASET_DIR = "data/dev_smoke" if SMOKE_RUN else "data/dev"
smoke_flag = "--smoke" if SMOKE_RUN else ""
!{sys.executable} -m wm_hw.dataset --config configs/dev.yaml --output-dir {DATASET_DIR} {smoke_flag}

## 5. Train The Starter World Model

In [ ]:
!{sys.executable} -m wm_hw.train --config configs/student.yaml --model student --dataset-dir {DATASET_DIR} --output-dir artifacts/student {smoke_flag}

## 6. Evaluate Test And OOD VPT Scoreboard

In [ ]:
!{sys.executable} -m wm_hw.eval_horizon --checkpoint-dir artifacts/student/best_checkpoint --dataset-dir {DATASET_DIR} --split test --horizon auto --eval-config configs/official_eval.yaml --output-dir artifacts/student/eval_test
!{sys.executable} -m wm_hw.eval_horizon --checkpoint-dir artifacts/student/best_checkpoint --dataset-dir {DATASET_DIR} --split ood --horizon auto --eval-config configs/official_eval.yaml --output-dir artifacts/student/eval_ood

## 7. Plot Diagnostics

In [ ]:
!{sys.executable} -m wm_hw.plotting --eval-dir artifacts/student/eval_test --output-dir artifacts/student/plots

import json
from IPython.display import Image, display

with open("artifacts/student/eval_test/metrics.json", "r", encoding="utf-8") as f:
    test_metrics = json.load(f)
with open("artifacts/student/eval_ood/metrics.json", "r", encoding="utf-8") as f:
    ood_metrics = json.load(f)

print("test VPT80@0.25:", test_metrics["VPT80@0.25"])
print("ood VPT80@0.25:", ood_metrics["VPT80@0.25"])
print("test max_horizon:", test_metrics["max_horizon"])
print("test nMSE_AUC:", test_metrics["nMSE_AUC"])
display(Image("artifacts/student/plots/survival_curve.png"))
display(Image("artifacts/student/plots/rollout_comparison.png"))

## 8. Optional Long-Horizon Scoreboard

The regular cells above stay lightweight. If you want the final public scoreboard artifacts, set `RUN_SCOREBOARD_DEMO = True` and `SMOKE_RUN = False` in the first cell. The public scoreboard config uses `warmup_steps=10` and `max_horizon=1000`.

In [ ]:
if RUN_SCOREBOARD_DEMO:
    import json
    assert not SMOKE_RUN, "Final scoreboard artifacts require SMOKE_RUN = False."
    SCOREBOARD_DIR = "data/public_scoreboard"
    run([sys.executable, "-m", "wm_hw.dataset", "--config", "configs/public_scoreboard.yaml", "--output-dir", SCOREBOARD_DIR])
    run([sys.executable, "-m", "wm_hw.eval_horizon", "--checkpoint-dir", "artifacts/student/best_checkpoint", "--dataset-dir", SCOREBOARD_DIR, "--split", "test", "--horizon", "auto", "--eval-config", "configs/official_eval.yaml", "--output-dir", "artifacts/student/public_scoreboard_test"])
    run([sys.executable, "-m", "wm_hw.plotting", "--eval-dir", "artifacts/student/public_scoreboard_test", "--output-dir", "artifacts/student/public_scoreboard_plots"])
    run(["bash", "-lc", "cd artifacts/student && zip -qr final_artifacts.zip best_checkpoint eval_test eval_ood plots public_scoreboard_test public_scoreboard_plots"])
    with open("artifacts/student/public_scoreboard_test/scoreboard_summary.json", "r", encoding="utf-8") as f:
        print(json.load(f))
else:
    print("Set RUN_SCOREBOARD_DEMO = True in the first cell to run the long-horizon scoreboard demo.")